# Estatística 2 - Aula prática 1_5 em Python

## UNIDADE Unidade 2: Analise de regressao  

## Secao 1.5: Testes para o modelo linear por MQO e a regressao stepwise   

By Jose P. Leitão

In [1]:
# Importar bibliotecas
import pandas as pd
pd.set_option('display.float_format', '{:.6f}'.format)

import numpy as np
import statsmodels.api as sm
from scipy.stats import norm, f

import warnings
warnings.filterwarnings('ignore')

In [2]:
# Usaremos o dataset "imoveiscwbav" que contem uma amostra de 541
# imoveis com preco dos imoveis e as caracteristicas de cada um
# destes imoveis.

# Vamos carregar o dataset
df = pd.read_csv('imoveiscwbav.csv')

In [3]:
# Vamos ver a base de dados
df.head()

,price,age,parea,tarea,bath,ensuit,garag,plaz,park,trans,kidca,school,health,bike,barb,balc,elev,fitg,party,categ
0,1100000.000000,15,150,190,4,1,2,0.080582,0.713281,2.386271,1.410981,0.902811,0.414647,0.213193,0,1,0,1,1,1
1,895000.000000,11,165,210,4,1,2,0.166351,0.698369,2.246304,1.862591,0.935579,0.256953,0.232553,0,0,0,0,0,1
2,2513600.000000,2,146,275,4,3,3,0.056075,1.312982,2.631411,1.591493,0.451791,0.232160,0.297093,0,1,0,1,1,1
3,755000.000000,25,163,238,3,1,2,0.321594,2.109958,2.138700,1.621586,0.447871,0.684845,0.347147,0,0,0,0,0,1
4,1099000.000000,1,107,189,3,1,2,0.146635,1.017530,1.797893,1.257243,0.884199,0.299009,0.778765,0,0,1,1,1,1


In [ ]:
# Vamos agora estimar o modelo linear por MQO com o preco em
# logaritmo neperiano que no "R" eh "log"; o "." quer dizer que 
# estamos regredindo o ln do preco contra todas as demais 
# variaveis, ou seja, todas as demais variaveis do dataset sao
# variaveis explicativas (Xs) no nosso modelo. Estamos guardando
# os resultados da estimativa em um objeto chamado "resultados"

# resultados <- lm(log(price)~., data=imoveis)

In [4]:
# Função que replica a função summary do R

def summary_lm_like_r(modelo):
    names = modelo.model.exog_names
    
    coef = modelo.params
    se = modelo.bse
    tvals = modelo.tvalues
    pvals = modelo.pvalues

    df = pd.DataFrame({
        "Estimate": coef,
        "Std. Error": se,
        "t value": tvals,
        "Pr(>|t|)": pvals
    }, index=names)

    # significance codes
    def signif(p):
        if p < 0.001:
            return '***'
        elif p < 0.01:
            return '**'
        elif p < 0.05:
            return '*'
        elif p < 0.1:
            return '.'
        else:
            return ''

    df[''] = df["Pr(>|t|)"].apply(signif)

    print("\nCoefficients:")
    print(df)

    print("\n---")

    print(f"Residual standard error: {np.sqrt(modelo.mse_resid):.4f} "
          f"on {int(modelo.df_resid)} degrees of freedom")

    print(f"Multiple R-squared: {modelo.rsquared:.4f}, "
          f"Adjusted R-squared: {modelo.rsquared_adj:.4f}")

    print(f"F-statistic: {modelo.fvalue:.4f} "
          f"on {int(modelo.df_model)} and {int(modelo.df_resid)} DF, "
          f"p-value: {modelo.f_pvalue:.4g}")

In [5]:
# Variavel Dependente convertida para o log natural
y = np.log(df['price'])

# Variaveis Independentes Todas as colunas, exceto a primeira (price)
X = df.iloc[:,1:]

# Adiciona constante para para criar um intercept
X = sm.add_constant(X)

# Usa o Método dos Mínimos Quadrados (Ordinary Least Square)
model = sm.OLS(y, X)

# O resultado, após o fit é salvo em objeto da classe OLSResults
resultados = model.fit()

# Usamos o método summary para ter um sumário
summary_lm_like_r(resultados)


Coefficients:
        Estimate  Std. Error    t value  Pr(>|t|)     
const  12.201319    0.098134 124.333730  0.000000  ***
age    -0.012660    0.000774 -16.363512  0.000000  ***
parea   0.004561    0.000471   9.685788  0.000000  ***
tarea   0.001635    0.000252   6.490408  0.000000  ***
bath    0.033863    0.011192   3.025549  0.002604   **
ensuit  0.057477    0.014005   4.103917  0.000047  ***
garag   0.147881    0.016417   9.008049  0.000000  ***
plaz    0.148085    0.071095   2.082917  0.037746    *
park   -0.069153    0.020490  -3.375038  0.000793  ***
trans   0.023339    0.017143   1.361465  0.173955     
kidca   0.027713    0.026334   1.052357  0.293123     
school -0.011268    0.042736  -0.263675  0.792134     
health  0.021321    0.042419   0.502634  0.615434     
bike   -0.060710    0.042311  -1.434862  0.151926     
barb    0.021269    0.017055   1.247075  0.212930     
balc    0.044797    0.019047   2.351904  0.019049    *
elev   -0.067616    0.019087  -3.542560  0.000432 

In [ ]:
###############################################################
#                       OUTLIERS                              #
###############################################################

# Vamos fazer o teste de Bonferroni sobre os resultados: 

In [6]:
# Função que replica a função Outlier_test do R
import scipy.stats as stats
from statsmodels.stats.multitest import multipletests

def outlier_test_py(modelo, alpha=0.05, n_max=10, only_extreme=True):
    """
    Replica a função outlierTest() do R (car package).

    Parâmetros:
    - modelo: objeto ajustado do statsmodels (OLSResults)
    - alpha: nível de significância
    - n_max: número máximo de outliers a retornar
    - only_extreme: se True, retorna apenas o mais extremo (default do R)

    Retorna:
    - DataFrame com:
        index: observações
        student_resid: resíduo studentizado externo
        unadj_p: p-valor sem ajuste
        bonf_p: p-valor ajustado (Bonferroni)
    """

    influence = modelo.get_influence()
    resid_student = influence.resid_studentized_external

    n = len(resid_student)
    df = int(modelo.df_resid)

    # p-values bicaudais
    p_values = 2 * (1 - stats.t.cdf(np.abs(resid_student), df=df))

    # ajuste de Bonferroni
    _, p_adj, _, _ = multipletests(p_values, method='bonferroni')

    # montar DataFrame
    results = pd.DataFrame({
        'student_resid': resid_student,
        'unadj_p': p_values,
        'bonf_p': p_adj
    })

    # ordenar pelo valor absoluto do resíduo (mais extremo primeiro)
    results['abs_resid'] = np.abs(results['student_resid'])
    results = results.sort_values(by='abs_resid', ascending=False)

    # filtrar significativos
    results_sig = results[results['bonf_p'] < alpha]

    if results_sig.empty:
        print("No Studentized residuals with Bonferroni p < {:.3f}".format(alpha))
        return None

    # limitar número de resultados
    if only_extreme:
        results_sig = results_sig.head(1)
    else:
        results_sig = results_sig.head(n_max)

    # remover coluna auxiliar
    results_sig = results_sig.drop(columns='abs_resid')

    return results_sig

In [7]:
outlier_test_py(resultados)

,student_resid,unadj_p,bonf_p
392,5.022930,0.000001,0.000379


In [ ]:
# O resultado eh:
#rstudent unadjusted p-value      Bonferroni p
#392  5.02293      0.00000070067   0.00037906

In [8]:
# Exibe a observação contendo o outlier
display(df.iloc[392])

price    4229891.000000
age            3.000000
parea        155.000000
tarea        266.000000
bath           3.000000
ensuit         3.000000
garag          3.000000
plaz           0.355986
park           2.331109
trans          1.481412
kidca          1.457472
school         0.620898
health         0.176508
bike           0.533282
barb           1.000000
balc           1.000000
elev           0.000000
fitg           1.000000
party          1.000000
categ          1.000000
Name: 392, dtype: float64

In [9]:
# Existe um outlier na linhas 392
# Neste caso, se vc julgar prudente, vc deve deletar a linha
# indicada na tabela de dados. Vamos editar o conjunto de 
# dados (eliminacao da linha 392):
df.drop([392], inplace=True)

In [10]:
# Depois deve-se refazer a estimativa dos resultados e o 
# teste de outliers, se necessario deve-se extrair novas
# observacoes e recalcular o modelo e o teste de outlier

# Variavel Dependente convertida para o log natural
y = np.log(df['price'])

# Variaveis Independentes Todas as colunas, exceto a primeira (price)
X = df.iloc[:,1:]

# Adiciona constante para para criar um intercept
X = sm.add_constant(X)

# Usa o Método dos Mínimos Quadrados (Ordinary Least Square)
model = sm.OLS(y, X)

# O resultado, após o fit é salvo em objeto da classe OLSResults
resultados = model.fit()

# Usamos o método summary para ter um sumário
summary_lm_like_r(resultados)


Coefficients:
        Estimate  Std. Error    t value  Pr(>|t|)     
const  12.219045    0.095993 127.290709  0.000000  ***
age    -0.012790    0.000757 -16.901366  0.000000  ***
parea   0.004558    0.000460   9.902366  0.000000  ***
tarea   0.001598    0.000246   6.483976  0.000000  ***
bath    0.039683    0.011002   3.606874  0.000340  ***
ensuit  0.052440    0.013727   3.820166  0.000149  ***
garag   0.143345    0.016073   8.918362  0.000000  ***
plaz    0.136923    0.069533   1.969190  0.049462    *
park   -0.072513    0.020040  -3.618344  0.000325  ***
trans   0.023522    0.016757   1.403672  0.161013     
kidca   0.027065    0.025743   1.051369  0.293578     
school -0.012687    0.041776  -0.303695  0.761482     
health  0.029839    0.041501   0.718999  0.472464     
bike   -0.060187    0.041360  -1.455182  0.146222     
barb    0.019031    0.016678   1.141128  0.254342     
balc    0.042828    0.018623   2.299733  0.021859    *
elev   -0.063588    0.018675  -3.404977  0.000713 

In [11]:
# Refazendo o teste de outliers
outlier_test_py(resultados)

,student_resid,unadj_p,bonf_p
363,-4.005752,0.000071,0.038253


In [12]:
# O resultado eh:
#rstudent unadjusted p-value Bonferroni p
#363 -4.005752        0.000070857     0.038263

# Ainda temos um outlier nas linhas 363, vamos retira-lo
df.drop([363], inplace=True)

In [13]:
# Vamos reestimar o modelo novamente 

# Variavel Dependente convertida para o log natural
y = np.log(df['price'])

# Variaveis Independentes Todas as colunas, exceto a primeira (price)
X = df.iloc[:,1:]

# Adiciona constante para para criar um intercept
X = sm.add_constant(X)

# Usa o Método dos Mínimos Quadrados (Ordinary Least Square)
model = sm.OLS(y, X)

# O resultado, após o fit é salvo em objeto da classe OLSResults
resultados = model.fit()

# Usamos o método summary para ter um sumário
summary_lm_like_r(resultados)


Coefficients:
        Estimate  Std. Error    t value  Pr(>|t|)     
const  12.241601    0.094801 129.129104  0.000000  ***
age    -0.013068    0.000749 -17.441375  0.000000  ***
parea   0.004595    0.000454  10.123578  0.000000  ***
tarea   0.001564    0.000243   6.435917  0.000000  ***
bath    0.039755    0.010846   3.665314  0.000272  ***
ensuit  0.052562    0.013533   3.884029  0.000116  ***
garag   0.139524    0.015874   8.789466  0.000000  ***
plaz    0.148993    0.068614   2.171457  0.030349    *
park   -0.079004    0.019823  -3.985497  0.000077  ***
trans   0.024126    0.016521   1.460354  0.144798     
kidca   0.026912    0.025378   1.060413  0.289450     
school -0.017776    0.041204  -0.431413  0.666347     
health  0.031674    0.040916   0.774126  0.439209     
bike   -0.055836    0.040789  -1.368894  0.171624     
barb    0.017534    0.016446   1.066190  0.286834     
balc    0.041224    0.018364   2.244837  0.025199    *
elev   -0.064724    0.018413  -3.515161  0.000478 

In [14]:
# Refazendo o teste de outliers
outlier_test_py(resultados)

No Studentized residuals with Bonferroni p < 0.050


In [ ]:
# O resultado eh:
# No Studentized residuals with Bonferroni p < 0.05

# O teste apresentou que nao temos mais outliers

In [ ]:
################################################################
#                   Regressao STEPWISE                         #
#             Escolha das variaveis e do modelo                #
################################################################

In [15]:
# Função que replica a função stepwise, no modo backward usando AIC como critério (R)
import statsmodels.api as sm

def stepwise_backward_aic(X, y, verbose=True):
    """
    Backward stepwise selection usando AIC como critério.

    Parâmetros:
    - X: DataFrame com variáveis explicativas
    - y: variável resposta
    - verbose: imprime progresso

    Retorna:
    - modelo final ajustado
    - lista de variáveis selecionadas
    """

    remaining = list(X.columns)
    best_aic = float('inf')
    current_model = None

    while len(remaining) > 0:
        aic_with_candidates = []

        # testa remover cada variável
        for candidate in remaining:
            vars_test = [v for v in remaining if v != candidate]
            
            X_test = sm.add_constant(X[vars_test])
            model = sm.OLS(y, X_test).fit()
            
            aic_with_candidates.append((model.aic, candidate, model))

        # encontra melhor remoção
        aic_with_candidates.sort()
        best_new_aic, worst_var, best_model_candidate = aic_with_candidates[0]

        if verbose:
            print(f"Removendo: {worst_var}, AIC: {best_new_aic:.3f}")

        # critério de parada
        if best_new_aic < best_aic:
            remaining.remove(worst_var)
            best_aic = best_new_aic
            current_model = best_model_candidate
        else:
            break

    if verbose:
        print("\nModelo final:")
        print("Variáveis:", remaining)
        print("AIC:", best_aic)

    return current_model, remaining

In [16]:
# Recria X e y com todas a colunas

# Variavel Dependente convertida para o log natural
y = np.log(df['price'])

# Variaveis Independentes Todas as colunas, exceto a primeira (price)
X = df.iloc[:,1:]

# stepwise(resultados, direction= 'backward', criterion ='AIC')
_, colunas_restantes = stepwise_backward_aic(X, y)

Removendo: school, AIC: -383.172
Removendo: health, AIC: -384.307
Removendo: kidca, AIC: -385.585
Removendo: barb, AIC: -386.094
Removendo: bike, AIC: -386.439
Removendo: plaz, AIC: -385.537

Modelo final:
Variáveis: ['age', 'parea', 'tarea', 'bath', 'ensuit', 'garag', 'plaz', 'park', 'trans', 'balc', 'elev', 'fitg', 'party', 'categ']
AIC: -386.4388571519062


In [17]:
# A regressao setpwise indicou que devem permanecer no modelo
# as seguintes variaveis dependentes: age, parea, tarea, bath,
# ensuit, garag, plaz, park, trans, balc, elev, fitg, party,
# categ.

# Vamos reestimar o modelo somente com essas variaveis

# Variavel Dependente convertida para o log natural
y = np.log(df['price'])

# Variaveis Independentes com as colunas indicadas
X = df[colunas_restantes]

# Adiciona constante para para criar um intercept
X = sm.add_constant(X)

# Usa o Método dos Mínimos Quadrados (Ordinary Least Square)
model = sm.OLS(y, X)

# O resultado, após o fit é salvo em objeto da classe OLSResults
resultados = model.fit()

# Usamos o método summary para ter um sumário
summary_lm_like_r(resultados)


Coefficients:
        Estimate  Std. Error    t value  Pr(>|t|)     
const  12.247897    0.077492 158.052792  0.000000  ***
age    -0.013323    0.000734 -18.158957  0.000000  ***
parea   0.004593    0.000449  10.239053  0.000000  ***
tarea   0.001568    0.000241   6.495430  0.000000  ***
bath    0.041320    0.010797   3.826862  0.000145  ***
ensuit  0.051187    0.013438   3.809212  0.000156  ***
garag   0.141409    0.015700   9.006798  0.000000  ***
plaz    0.106343    0.063234   1.681737  0.093216    .
park   -0.077085    0.015912  -4.844419  0.000002  ***
trans   0.028559    0.014580   1.958707  0.050677    .
balc    0.043227    0.017933   2.410482  0.016275    *
elev   -0.064619    0.018354  -3.520632  0.000468  ***
fitg    0.068129    0.020408   3.338374  0.000903  ***
party   0.057996    0.020608   2.814282  0.005072   **
categ   0.282487    0.039143   7.216861  0.000000  ***

---
Residual standard error: 0.1668 on 524 degrees of freedom
Multiple R-squared: 0.8994, Adjusted R-squ

In [ ]:
# Agora temos quase todas as variaveis estatisticamente 
# significativas, excecao para as variaveis "trans" e "plaz",
# que sao significativas apenas a 90% de confianca, mas ainda
# devemos fazer outros testes/ajustes e essas variaveis podem
# se tornar significativas a 95% de confianca.
# Se necessario faremos a exclusao mais adiante.



In [ ]:
##############################################################
#                       Multicolinearidade                   #
##############################################################

In [ ]:
# Vamos verificar a existencia de multicolineariedade, que eh 
# identificar a alta correlacao entre as variaveis. Vamos 
# fazer isto com o Fator de Inflacao da variancia (VIF), que 
# eh uma medida da correlacao multipla das variaveis e sua
# influencia no R2, R2 ajustado e variancia.

In [ ]:
# Vamos calcular o VIF 
#car::vif(resultados)

In [18]:
# Função que replica o comportamento da função do vif do R

from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm

def vif_py_r_like(X):
    X_const = sm.add_constant(X, has_constant='skip')

    vif_data = pd.DataFrame()
    vif_data["variable"] = X_const.columns
    vif_data["VIF"] = [
        variance_inflation_factor(X_const.values, i)
        for i in range(X_const.shape[1])
    ]

    # remover intercepto (igual R)
    vif_data = vif_data[vif_data["variable"] != "const"]

    return vif_data.reset_index(drop=True)

In [21]:
vif_py_r_like(X)

,variable,VIF
0,age,1.679947
1,parea,4.074078
2,tarea,3.681996
3,bath,3.046737
4,ensuit,2.866718
5,garag,2.152874
6,plaz,1.126224
7,park,1.608973
8,trans,1.479620
9,balc,1.540686


In [ ]:
# Aqui a regra nao eh unica, muitos pesquisadores usam valores de
# corte do VIF diferentes, alguns usam valores acima de 4, outros
# acima de 5, outros acima de 10; recomenda-se o uso de valores 
# acima de 5, que eh o que a maioria dos pesquisadores utiliza.

# Portanto, como nao existe nenhuma variavel com VIF acima de 5, 
# entao nao existe nenhuma variavel para retirar do modelo.

In [ ]:
###############################################################
#                       NORMALIDADE                           #
###############################################################

In [ ]:
# Vamos testar a normalidade dos residuos da regressao.
# Vamos utilizar o teste de Shapiro-Wilk para amostras de ate
# 5000 observacoes

# As hipoteses do teste:
# H0: A amostra eh normalmente distribuida
# Ha: A amostra nao eh normalmente distribuida

In [22]:
# obter os resíduos do modelo
residuos = resultados.resid

In [23]:
from scipy.stats import shapiro

stat, p_value = shapiro(residuos)

print(f"Estatística W: {stat:.4f}")
print(f"p-valor: {p_value:.4f}")

Estatística W: 0.9964
p-valor: 0.2629


In [ ]:
# O resultado:
# Como o p-value > 0.05 entao nao rejeitamos H0, ou seja, a 
# amostra eh normalmente distribuida.

In [ ]:
# Vamos utilizar o teste de normalidade de Kolmogorov-Smirnov
# para amostras de qualquer tamanho:
#lillie.test(resultados$residuals)


In [24]:
from scipy.stats import kstest

# padronizar (ESSENCIAL para normal)
residuos_std = (residuos - np.mean(residuos)) / np.std(residuos, ddof=1)

# teste KS contra normal padrão
stat, p_value = kstest(residuos_std, 'norm')

print(f"KS statistic: {stat:.4f}")
print(f"p-value: {p_value:.4f}")

KS statistic: 0.0275
p-value: 0.7997


In [ ]:
# O resultado:
# Como o p-value > 0.05 entao nao rejeitamos H0, ou seja, a 
# amostra eh normalmente distribuida.

# Caso a amostra nao fosse normalmente distribuida, deveriamos
# optar por um modelo de regressao nao parametrico ou 
# semiparametrico. Pode-se ainda tentar normalizar o modelo
# com transformacao Box-Cox.

In [ ]:
#################################################################
#                  ESPECIFICACAO DO MODELO                      #
#################################################################

In [ ]:
# Precisamos fazer o teste RESET de especificacao do modelo.
# As hipoteses do teste sao: 
# H0 = O modelo esta corretamente especificado; 
# HA = O modelo NAO esta corretamente especificado;

In [ ]:
# Atencao, as variaveis fatores (factor) ficam fora do teste
# resettest(log(price)~age+parea+tarea+bath+ensuit+
#             garag+plaz+park+trans, power=2:3, 
#           type=c("fitted", "regressor", "princomp"), 
#           data=imoveis)


In [25]:
# Função que replica o comportamento do resettest do R

from statsmodels.stats.diagnostic import linear_reset

def resettest_like_r(
    model,
    test_types=("fitted",),
    power=2,
    use_f=True,
    verbose=True
):
    """
    Wrapper para simular o comportamento do resettest() do R.

    Parameters
    ----------
    model : statsmodels regression results (ex: OLSResults)
    test_types : iterable of str
        Opções: "fitted", "exog", "princomp"
    power : int ou iterable
        Potências usadas no teste (ex: 2 ou [2,3])
    use_f : bool
        True -> F-test | False -> Chi-square
    verbose : bool
        Se True, imprime resultados formatados

    Returns
    -------
    dict
        Resultados organizados por tipo de teste
    """

    # garantir lista de potências
    if isinstance(power, int):
        power = [power]

    results = {}

    for t in test_types:
        res = linear_reset(
            model,
            test_type=t,
            power=power,
            use_f=use_f
        )

        results[t] = {
            "statistic": res.statistic,
            "pvalue": res.pvalue,
            "df_denom": getattr(res, "df_denom", None),
            "df_num": getattr(res, "df_num", None),
            "distribution": "F" if use_f else "chi2"
        }

    if verbose:
        print("Ramsey RESET test\n")
        for t, r in results.items():
            print(f"[{t}]")
            print(f"  Test statistic ({r['distribution']}): {r['statistic']:.4f}")
            print(f"  p-value: {r['pvalue']:.6f}")
            if r["df_num"] is not None:
                print(f"  df_num: {r['df_num']}")
            if r["df_denom"] is not None:
                print(f"  df_denom: {r['df_denom']}")
            print("")

    return results

In [26]:
# Variavel Dependente convertida para o log natural
y = np.log(df['price'])

# Variaveis explicativas, excluindo as binárias
colunas = ['age', 'parea', 'tarea', 'bath', 'ensuit', 'garag', 'plaz', 'park', 'trans']
X = df[colunas]

# Adiciona constante para para criar um intercept
X = sm.add_constant(X)

# Usa o Método dos Mínimos Quadrados (Ordinary Least Square)
model = sm.OLS(y, X)

# O resultado, após o fit é salvo em objeto da classe OLSResults
resultados = model.fit()

In [27]:
reset = resettest_like_r(
    resultados,
    test_types=["fitted", "exog", "princomp"],
    power=[2, 3],
    use_f=True
)

Ramsey RESET test

[fitted]
  Test statistic (F): 0.0805
  p-value: 0.922626
  df_num: 2.0
  df_denom: 527.0

[exog]
  Test statistic (F): 4.0256
  p-value: 0.000000
  df_num: 16
  df_denom: 511.0

[princomp]
  Test statistic (F): 2.6338
  p-value: 0.072749
  df_num: 2.0
  df_denom: 527.0



In [ ]:
# O resultado do teste eh:
# RESET test
# data:  log(price) ~ age + parea + tarea + bath + ensuit + garag 
#                     + plaz + park + trans
# RESET = 0.080544, df1 = 2, df2 = 527, p-value = 0.9226

In [28]:
# O valor tabelado para compararmos com o valor do teste eh:
#qf(.95,df1=2,df2=527)
f.ppf(0.95, dfn=2, dfd=527)

np.float64(3.012826237080222)

In [ ]:
# Resposta:
# F tabelado = 3,012826

In [ ]:
# Como o F calculado (0,08) eh menor que o F tabelado (3,01) 
# nao existe erro de especificacao do modelo.
# Regra de bolso: Toda vez que p-value > 0.05 nao existe erro
# de especificacao do modelo, não rejeitamos H0.

# Caso houvesse algum erro de especificacao do modelo teriamos 
# de incluir (testar) as potencias 2 e 3 nas variaveis (exog), alem 
# outras formas de transformacao das variaveis, uma a uma e
# testar cada modelo pelo teste "reset".

In [ ]:
################################################################
#                   AUTOCORRELACAO DOS RESIDUOS                #
################################################################

In [ ]:
# Verificando a autocorrelacao dos residuos - empregavel somente
# para dados de series temporais, vamos fazer aqui soh para 
# exemplificar, caso voce precise fazer quando a sua amostra 
# for de dados de series temporais


In [ ]:
# TABELA DE ANALISE DAS HIPOTESES
# HIPOTESE NULA            DECISAO       SE
# nao existe autocorr(+)   rejeitar      0 < d < dl
# nao existe autocorr +    sem decisao   dl <= d <= du
# nao existe autocorr -    rejeitar      4 - dl < d < 4
# nao existe autocorr -    sem decisao   4 - du <= d <= 4 - dl
# nenhuma autocorr(+)ou(-) nao rejeitar  du < d < 4-du ***

In [ ]:
# As hipoteses do teste:
# H0: Nao existe autocorrelacao nos residuos
# Ha: Existe autocorrelacao nos residuos ou o teste eh nao
#     conclusivo

In [29]:
# O teste:
# dwtest(log(price)~age+parea+tarea+bath+ensuit+
#          garag+plaz+park+trans+balc+elev+fitg+
#          party+categ, alternative="greater", data=imoveis)

# Variavel Dependente convertida para o log natural
y = np.log(df['price'])

# Variaveis Independentes com as colunas indicadas
X = df[colunas_restantes]

# Adiciona constante para para criar um intercept
X = sm.add_constant(X)

# Usa o Método dos Mínimos Quadrados (Ordinary Least Square)
model = sm.OLS(y, X)

# O resultado, após o fit é salvo em objeto da classe OLSResults
resultados = model.fit()

# resíduos
residuals = resultados.resid

# Durbin-Watson
from statsmodels.stats.stattools import durbin_watson

dw = durbin_watson(residuals)

print(f"{dw:.4f}")

1.9661


In [ ]:
# Na implementação de statsmodels para o teste de Durbin-Watson, não apresenta o p-value apenas a estatística DW
# para interpretar o resultado DW:
# ~2 -> não existe correlação entre os resíduos
# <1 -> autocorrelação positiva
# >3 -> autocorrelação negativa

#DW = 1.9661, p-value = 0.3191
# alternative hypothesis: true autocorrelation is greater than 0

In [ ]:
# Parametros do teste:
# k = Número de variáveis explanatórias, excluindo a constante: 14
# n = Tamanho da amostra: 539

# k = 14,  n = 539 (maximo é 200 no livro)
# dl = 1,528, du = 1,824 (tabela no livro do Gujarati)

# 4 - dl = 2,472
# 4 - du = 2,176

![image info](./estatistica_d_de_Durbin-Watson.png)

In [ ]:
# O valor de 1.9661 esta entre 1,824 e 2,176, ultima linha da
# tabela de analise, portanto nao existe autocorrelacao nos
# residuos.
# Toda vez que a estatistica DW estiver muito proxima de 2
# quer dizer que nao existe autocorrelacao nos residuos.

# Regra de bolso = Toda vez que o p-value > 0.05 nao existe
#                  autocorrelacao nos residuos

# Caso exista autocorrelacao positiva ou negativa, utilizar o 
# metodo da primeira diferenca exposto na secao 12.9 "metodo 
# da primeira diferenca" do livro do Gujarati e Porter.
# Alternativamente, pode-se utilizar estimadores sandwich 
# HAC.

In [ ]:
###############################################################
#                    HETEROCEDASTICIDADE                      #
###############################################################

In [ ]:
# Verificando a heterocedasticidade = variancia nao constante
# (amostra homogenea?)

# Para grandes amostras utilizar o teste de Breusch-Pagan, 
# caso contrario utilizar o teste de Goldfeld-Quandt

# Vamos fazer primeiro o teste de Goldfeld-Quandt soh para
# exemplificar o uso da funcao de teste (nao deve ser usada
# para grandes amostras)
# As hipoteses do teste:
# H0: A amostra eh homocedastica
# Ha: A amostra eh heterocedastica

In [ ]:
# gqtest(resultados, order.by = ~age+parea+tarea+bath+ensuit+
#          garag+plaz+park+trans+balc+elev+fitg+
#          party+categ, data = imoveis, fraction = 5)

In [30]:
from statsmodels.stats.diagnostic import het_goldfeldquandt

gq_test = het_goldfeldquandt(resultados.resid, resultados.model.exog, drop=5)

print(f"GQ : {gq_test[0]:.3f}, p-value : {gq_test[1]:.4f}, Altenative hypothesis : {gq_test[2]}")

GQ : 1.212, p-value : 0.0638, Altenative hypothesis : increasing


In [31]:
# O valor tabelado para compararmos com o valor do teste eh:
#qf(.95,df1=252,df2=252)
print(f"{f.ppf(0.95, dfn=252, dfd=252):.5f}")

1.23075


In [ ]:
# Como o valor da estatistica de teste (1.206) eh menor que 
# a estatistica tabelada (1.23075) não rejeitamos H0, ou seja,
# se fosse uma amostra pequena, poderiamos concluir que a 
# amostra eh homogenea (homocedastica).
# Regra de bolso: como p-value > 0.05 a amostra eh 
# homocedastica 

# Agora vamos utilizar um teste adequado para grandes amostras
# que eh o nosso caso. Vamos utilizar o teste de Breusch-Pagan. 

In [ ]:
# As hipoteses do teste:
# H0: O modelo eh homocedastico (variancia constante) 
# HA: O modelo eh Heterocedatico (variancia nao-constante)

# O teste:
# bptest(log(price)~age+parea+tarea+bath+ensuit+
#          garag+plaz+park+trans+balc+elev+fitg+
#          party+categ, studentize=FALSE, data=imoveis)

# O resultado:

In [45]:
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan

# modelo
X = df[colunas_restantes]
X = sm.add_constant(X)
y = np.log(df['price'])

model = sm.OLS(y, X)

resultados = model.fit()

# resíduos
residuals = resultados.resid

# teste Breusch-Pagan
bp_test = het_breuschpagan(residuals, model.exog)

print(f"BP: {bp_test[0]:.4f}, p-value: {bp_test[1]:.4f}, Fvalue: {bp_test[2]:.4f}, Fp-value: {bp_test[3]:.4f}")

BP: 48.3084, p-value: 0.0000, Fvalue: 3.6848, Fp-value: 0.0000


In [ ]:
# O resultado:
# Breusch-Pagan test
#  data:  lnprice~age+parea+tarea+bath+ensuit+garag+plaz+
#                 park+trans+balc+elev+fitg+party+categ
# BP = 59.988, df = 14, p-value = 0.0000001179

In [46]:
# Obtendo o valor critico (tabelado) da estatistica 
# qui-quadrado:
# qchisq(0.95, df=14)
from scipy.stats import chi2

print(f"Valor critico: {chi2.ppf(0.95, df=14):.5f}")

Valor critico: 23.68479


In [ ]:
# O resultado eh confrontado em um teste qui-quadrado com
# k-1 graus de liberdade. Como o resultado do teste BP 
# (59.988) eh maior que o tabelado (23,68478) rejeita-se a
# hipotese de que o modelo eh homocedastico.
# Regra de bolso: se p-value < 0.05 existe 
# heterocedasticidade, temos um problema importante e
# precisamos corrigi-lo.

In [47]:
# Vamos refazer a regressao novamente para lembrar do modelo:
# resultados <- lm(log(price)~age+parea+tarea+bath+ensuit+
#                    garag+plaz+park+trans+balc+elev+fitg+
#                    party+categ, data=imoveis)
# summary(resultados)
# modelo
# matriz de regressoras (mesmas do modelo)
X = df[colunas_restantes]
X = sm.add_constant(X)
y = np.log(df['price'])

model = sm.OLS(y, X)

resultados = model.fit()

summary_lm_like_r(resultados)


Coefficients:
        Estimate  Std. Error    t value  Pr(>|t|)     
const  12.247897    0.077492 158.052792  0.000000  ***
age    -0.013323    0.000734 -18.158957  0.000000  ***
parea   0.004593    0.000449  10.239053  0.000000  ***
tarea   0.001568    0.000241   6.495430  0.000000  ***
bath    0.041320    0.010797   3.826862  0.000145  ***
ensuit  0.051187    0.013438   3.809212  0.000156  ***
garag   0.141409    0.015700   9.006798  0.000000  ***
plaz    0.106343    0.063234   1.681737  0.093216    .
park   -0.077085    0.015912  -4.844419  0.000002  ***
trans   0.028559    0.014580   1.958707  0.050677    .
balc    0.043227    0.017933   2.410482  0.016275    *
elev   -0.064619    0.018354  -3.520632  0.000468  ***
fitg    0.068129    0.020408   3.338374  0.000903  ***
party   0.057996    0.020608   2.814282  0.005072   **
categ   0.282487    0.039143   7.216861  0.000000  ***

---
Residual standard error: 0.1668 on 524 degrees of freedom
Multiple R-squared: 0.8994, Adjusted R-squ

In [48]:
# Vamos fazer a correcao da heterocedasticidade por
# estimadores sandwich, isto eh feito pela seguinte funcao:
#coeftest(resultados, vcov=vcovHC(resultados, type="HC"))

robust_results = resultados.get_robustcov_results(cov_type='HC0')  # equivalente ao type="HC"
summary_lm_like_r(robust_results)


Coefficients:
        Estimate  Std. Error    t value  Pr(>|t|)     
const  12.247897    0.073471 166.703459  0.000000  ***
age    -0.013323    0.000779 -17.093429  0.000000  ***
parea   0.004593    0.000548   8.383468  0.000000  ***
tarea   0.001568    0.000295   5.321163  0.000000  ***
bath    0.041320    0.009985   4.138316  0.000041  ***
ensuit  0.051187    0.013778   3.715112  0.000225  ***
garag   0.141409    0.017452   8.102752  0.000000  ***
plaz    0.106343    0.068502   1.552408  0.121168     
park   -0.077085    0.015759  -4.891635  0.000001  ***
trans   0.028559    0.014901   1.916538  0.055840    .
balc    0.043227    0.016677   2.592081  0.009806   **
elev   -0.064619    0.017129  -3.772471  0.000180  ***
fitg    0.068129    0.018203   3.742786  0.000202  ***
party   0.057996    0.019645   2.952275  0.003296   **
categ   0.282487    0.041763   6.764101  0.000000  ***

---
Residual standard error: 0.1668 on 524 degrees of freedom
Multiple R-squared: 0.8994, Adjusted R-squ

In [ ]:
# Analisando os resultados vemos que as variaveis "plaz" e 
# "trans", nao sao significativas a 95% de confianca, 
# portanto vamos retira-las do modelo, e reestimar o modelo
# e o teste de heterocedasticidade.

In [53]:
# Reestimando o modelo sem as variaveis "plaz" e "trans":
remover = ["plaz", "trans"]
novas_colunas = [c for c in colunas_restantes if c not in remover]

X = df[novas_colunas]
X = sm.add_constant(X)
y = np.log(df['price'])

model = sm.OLS(y, X)

resultados1 = model.fit()

summary_lm_like_r(resultados1)


Coefficients:
        Estimate  Std. Error    t value  Pr(>|t|)     
const  12.337292    0.066285 186.124958  0.000000  ***
age    -0.013356    0.000737 -18.130467  0.000000  ***
parea   0.004732    0.000447  10.595620  0.000000  ***
tarea   0.001551    0.000242   6.398716  0.000000  ***
bath    0.041940    0.010842   3.868447  0.000123  ***
ensuit  0.048834    0.013466   3.626460  0.000315  ***
garag   0.141587    0.015763   8.982131  0.000000  ***
park   -0.084251    0.013514  -6.234238  0.000000  ***
balc    0.041831    0.017995   2.324594  0.020473    *
elev   -0.064657    0.018331  -3.527149  0.000457  ***
fitg    0.063758    0.020361   3.131404  0.001836   **
party   0.059770    0.020668   2.891919  0.003987   **
categ   0.273204    0.039035   6.998955  0.000000  ***

---
Residual standard error: 0.1675 on 526 degrees of freedom
Multiple R-squared: 0.8981, Adjusted R-squared: 0.8957
F-statistic: 386.2201 on 12 and 526 DF, p-value: 9.792e-252


In [54]:
# Refazendo o teste de heterocedasticidade:
# bptest(log(price)~age+parea+tarea+bath+ensuit+
#          garag+park+balc+elev+fitg+party+categ,
#        studentize=FALSE, data=imoveis)

# resíduos
residuals = resultados1.resid

# teste Breusch-Pagan
bp_test = het_breuschpagan(residuals, model.exog)

print(f"BP: {bp_test[0]:.4f}, p-value: {bp_test[1]:.4f}, Fvalue: {bp_test[2]:.4f}, Fp-value: {bp_test[3]:.4f}")


BP: 44.4272, p-value: 0.0000, Fvalue: 3.9375, Fp-value: 0.0000


In [ ]:
# O resultado:
# Breusch-Pagan test
# data:  lnprice ~ age + parea + tarea + bath + ensuit + 
#                  garag + park + balc + elev + fitg +
#                  party + categ
# BP = 55.195, df = 12, p-value = 0.000000167


In [55]:
# Obtendo o valor critico (tabelado) da estatistica 
# qui-quadrado:
# qchisq(0.95, df=12)
from scipy.stats import chi2

print(f"Valor critico: {chi2.ppf(0.95, df=12):.5f}")

Valor critico: 21.02607


In [ ]:
# Percebemos que melhorou o problema da heterocedasticidade,
# o valor da estatistica BP esta menor, portanto essas duas
# variaveis extraidas eram fontes de heterocedasticidade.
# Porem, ainda persiste o problema de heterocedasticidade
# pois p-value < 0.05

# Vamos novamente corrigir a heterocedasticidade por 
# regressao linear robusta (sandwich) e vamos guardar os 
# valores estimados no objeto "resultados1"
# resultados1 <- coeftest(resultados, vcov=vcovHC(resultados,
#                                                 type="HC"))

robust_results_2 = resultados1.get_robustcov_results(cov_type='HC0')  # equivalente ao type="HC"

# Visualizando os resultados
summary_lm_like_r(robust_results_2)

# Agora todas as variaveis sao significativas a 95% de 
# confianca, o que nos permite fazer inferencia com maior
# confiabilidade.



Coefficients:
        Estimate  Std. Error    t value  Pr(>|t|)     
const  12.337292    0.062644 196.941950  0.000000  ***
age    -0.013356    0.000786 -16.995737  0.000000  ***
parea   0.004732    0.000545   8.687033  0.000000  ***
tarea   0.001551    0.000302   5.137529  0.000000  ***
bath    0.041940    0.010199   4.112175  0.000045  ***
ensuit  0.048834    0.013878   3.518864  0.000471  ***
garag   0.141587    0.017709   7.995150  0.000000  ***
park   -0.084251    0.012931  -6.515442  0.000000  ***
balc    0.041831    0.016807   2.488985  0.013119    *
elev   -0.064657    0.016803  -3.847895  0.000134  ***
fitg    0.063758    0.018107   3.521237  0.000467  ***
party   0.059770    0.019522   3.061717  0.002313   **
categ   0.273204    0.041438   6.593159  0.000000  ***

---
Residual standard error: 0.1675 on 526 degrees of freedom
Multiple R-squared: 0.8981, Adjusted R-squared: 0.8957
F-statistic: 465.7132 on 12 and 526 DF, p-value: 4.528e-271


In [ ]:
############################################################
#                        ANOVA                             #
############################################################

In [63]:
colunas_restantes

['age',
 'parea',
 'tarea',
 'bath',
 'ensuit',
 'garag',
 'plaz',
 'park',
 'trans',
 'balc',
 'elev',
 'fitg',
 'party',
 'categ']

In [64]:
# Primeiro vamos estimar a ANOVA do modelo sem ajuste para
# heterocedasticidade, para o caso se estivermos trabalhando
# em um modelo sem heterocedasticidade
#car::Anova(resultados)
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm
# modelo

resultados = ols('np.log(price)~age+parea+tarea+bath+ensuit+\
                   garag+plaz+park+trans+balc+elev+fitg+party+categ', data=df).fit()


anova_table = anova_lm(resultados)

print(anova_table)

                 df    sum_sq   mean_sq           F   PR(>F)
age        1.000000 55.935282 55.935282 2011.150772 0.000000
parea      1.000000 56.364496 56.364496 2026.583124 0.000000
tarea      1.000000  5.776207  5.776207  207.683275 0.000000
bath       1.000000  3.637400  3.637400  130.782583 0.000000
ensuit     1.000000  0.287770  0.287770   10.346750 0.001377
garag      1.000000  2.932453  2.932453  105.436212 0.000000
plaz       1.000000  0.002695  0.002695    0.096913 0.755690
park       1.000000  1.832756  1.832756   65.896678 0.000000
trans      1.000000  0.001682  0.001682    0.060459 0.805869
balc       1.000000  0.648787  0.648787   23.327104 0.000002
elev       1.000000  0.030735  0.030735    1.105080 0.293638
fitg       1.000000  0.971402  0.971402   34.926726 0.000000
party      1.000000  0.369146  0.369146   13.272618 0.000296
categ      1.000000  1.448565  1.448565   52.083080 0.000000
Residual 524.000000 14.573789  0.027813         NaN      NaN


In [65]:
# Agora vamos estimar a ANOVA do modelo com ajuste para 
# heterocedasticidade 
#car::Anova(resultados,white.adjust=TRUE)
# ANOVA com ajuste de White (heterocedasticidade)
anova_table = anova_lm(resultados, typ=2, robust='hc3')

print(anova_table)

            sum_sq         df          F   PR(>F)
age       7.615792   1.000000 273.825499 0.000000
parea     1.782724   1.000000  64.097758 0.000000
tarea     0.711825   1.000000  25.593621 0.000001
bath      0.443589   1.000000  15.949234 0.000074
ensuit    0.356287   1.000000  12.810285 0.000377
garag     1.683149   1.000000  60.517545 0.000000
plaz      0.062852   1.000000   2.259827 0.133371
park      0.621503   1.000000  22.346099 0.000003
trans     0.095704   1.000000   3.441035 0.064157
balc      0.174981   1.000000   6.291429 0.012433
elev      0.372924   1.000000  13.408472 0.000276
fitg      0.365114   1.000000  13.127668 0.000319
party     0.227119   1.000000   8.166071 0.004438
categ     1.122355   1.000000  40.354242 0.000000
Residual 14.573789 524.000000        NaN      NaN


In [ ]:
# A ANOVA nos fornece a analise da variancia de cada 
# variavel do modelo 

#############################################################
# USAREMOS TUDO O QUE FIZEMOS ATE AQUI NA PROXIMA SECAO DE  #
# ESTUDOS. ENTAO EH IMPORTANTE QUE VOCE TENHA "TODOS" OS    #
# COMANDOS EXECUTADOS NESTA ROTINA PARA QUE POSSA EXECUTAR  #
# OS COMANDOS DA PROXIMA SECAO DE ESTUDOS E REPRODUZIR OS   #
# RESULTADOS.                                               #
#############################################################